# ETL validation
Ejecuta el pipeline en dry-run y lo compara con lo que hay en Postgres. **No escribe nada**; para cargar usa `python -m etl.main --load`.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from etl.main import run
from etl.db import get_engine

engine = get_engine()
production, reserves, prices = run(load=False)

In [ ]:
Q = """
SELECT m.name AS mineral_name, c.name AS country_name, p.year, p.tonnes AS tonnes_db
FROM {t} p
JOIN minerals m ON m.mineral_id = p.mineral_id
JOIN countries c ON c.country_id = p.country_id
"""
for table, new in [('production', production), ('reserves', reserves)]:
    db = pd.read_sql(Q.format(t=table), engine)
    cmp = db.merge(new, on=['mineral_name', 'country_name', 'year'], how='outer', indicator=True)
    print(f"=== {table} ===")
    print(cmp['_merge'].value_counts().to_string())
    both = cmp[cmp['_merge'] == 'both']
    diff = both[(both['tonnes_db'].astype(float) - both['tonnes'].astype(float)).abs() > 0.01]
    print('Valores distintos:', len(diff))

In [ ]:
db_prices = pd.read_sql("""
SELECT m.name AS mineral_name, pr.price_date, pr.price_usd AS price_db
FROM prices pr JOIN minerals m ON m.mineral_id = pr.mineral_id""", engine)
db_prices['price_date'] = pd.to_datetime(db_prices['price_date'])
new = prices.assign(price_date=pd.to_datetime(prices['price_date']))
cmp = db_prices.merge(new, on=['mineral_name', 'price_date'], how='outer', indicator=True)
print(cmp['_merge'].value_counts().to_string())
both = cmp[cmp['_merge'] == 'both']
print('Valores distintos:', ((both['price_db'].astype(float) - both['price_usd'].astype(float)).abs() > 0.01).sum())